In [ ]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, precision_recall_curve, roc_curve, auc

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Conv1D, MaxPooling1D, BatchNormalization,
    Activation, Flatten, Dense, Dropout
)
from tensorflow.keras.optimizers import Adamax
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.initializers import GlorotUniform

from tensorflow.keras.layers import ELU

SEED = 0
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

print("TensorFlow version:", tf.__version__)
print("GPU available:", tf.config.list_physical_devices('GPU'))

In [ ]:
CLOSE_DIR = ''   # each subfolder = 1 monitored label
OPEN_DIR  = ''   # all samples = unmonitored/unknown

LENGTH     = 5000   # sequence length after padding/truncation
NB_EPOCH   = 30
BATCH_SIZE = 128
VERBOSE    = 2
OPTIMIZER  = Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999, epsilon=1e-8)

# Closed-world split ratios
TRAIN_RATIO = 0.70
VALID_RATIO = 0.10
TEST_RATIO  = 0.20

# Open-world unknown sample counts
N_UNKNOWN_TRAIN = 400
N_UNKNOWN_TEST  = 10000

In [ ]:
def parse_trace(filepath, length=5000):
    """
    Read a trace file with two columns: timestamp  packet_size_with_direction
    e.g.: 2.1  -76  → incoming (direction = -1)
          1.5   88  → outgoing (direction = +1)

    Returns: numpy array of shape (length,) containing only directions (+1 or -1).
    """
    directions = []
    try:
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                parts = line.split()
                if len(parts) < 2:
                    continue
                pkt_size = float(parts[1])
                if pkt_size > 0:
                    directions.append(1.0)
                elif pkt_size < 0:
                    directions.append(-1.0)
                # skip pkt_size == 0
    except Exception:
        pass  # return zero vector on error

    seq = np.array(directions[:length], dtype=np.float32)
    if len(seq) < length:
        seq = np.pad(seq, (0, length - len(seq)), 'constant')
    return seq


def load_dataset_from_dir(root_dir, length=5000, label_map=None, fixed_label=None):
    """
    Load full dataset from root_dir.
    - Each subfolder is one label (folder name = label name).
    - label_map: dict {folder_name: int_label} — auto-created if None.
    - fixed_label: if set, all samples receive this label (for unmonitored traffic).

    Returns: X (N, length), y (N,), label_map
    """
    X, y = [], []

    if fixed_label is not None:
        # Flat read — all files under root_dir including subfolders
        all_files = [
            os.path.join(dirpath, fname)
            for dirpath, _, filenames in os.walk(root_dir)
            for fname in filenames
            if fname.endswith('.txt') or '.' not in fname
        ]
        for fpath in all_files:
            X.append(parse_trace(fpath, length))
            y.append(fixed_label)
        return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), None

    # Read per subfolder as label
    subdirs = sorted([
        d for d in os.listdir(root_dir)
        if os.path.isdir(os.path.join(root_dir, d))
    ])

    if label_map is None:
        label_map = {name: idx for idx, name in enumerate(subdirs)}

    for folder_name in subdirs:
        label = label_map.get(folder_name)
        if label is None:
            continue
        folder_path = os.path.join(root_dir, folder_name)
        for fname in os.listdir(folder_path):
            fpath = os.path.join(folder_path, fname)
            if os.path.isfile(fpath):
                X.append(parse_trace(fpath, length))
                y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.int32), label_map

In [ ]:
def build_dfnet(input_shape, classes):
    model = Sequential()

    filter_num      = [None, 32,  64,  128, 256]
    kernel_size     = [None,  8,   8,    8,   8]
    conv_stride     = [None,  1,   1,    1,   1]
    pool_stride     = [None,  4,   4,    4,   4]
    pool_size_list  = [None,  8,   8,    8,   8]

    # Block 1 — ELU
    model.add(Conv1D(filter_num[1], kernel_size[1], strides=conv_stride[1],
                     padding='same', input_shape=input_shape, name='block1_conv1'))
    model.add(BatchNormalization())
    model.add(ELU(alpha=1.0))
    model.add(Conv1D(filter_num[1], kernel_size[1], strides=conv_stride[1],
                     padding='same', name='block1_conv2'))
    model.add(BatchNormalization())
    model.add(ELU(alpha=1.0))
    model.add(MaxPooling1D(pool_size=pool_size_list[1], strides=pool_stride[1],
                           padding='same', name='block1_pool'))
    model.add(Dropout(0.1))

    # Block 2 — ReLU
    model.add(Conv1D(filter_num[2], kernel_size[2], strides=conv_stride[2],
                     padding='same', name='block2_conv1'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv1D(filter_num[2], kernel_size[2], strides=conv_stride[2],
                     padding='same', name='block2_conv2'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling1D(pool_size=pool_size_list[2], strides=pool_stride[2],
                           padding='same', name='block2_pool'))
    model.add(Dropout(0.1))

    # Block 3 — ReLU
    model.add(Conv1D(filter_num[3], kernel_size[3], strides=conv_stride[3],
                     padding='same', name='block3_conv1'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv1D(filter_num[3], kernel_size[3], strides=conv_stride[3],
                     padding='same', name='block3_conv2'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling1D(pool_size=pool_size_list[3], strides=pool_stride[3],
                           padding='same', name='block3_pool'))
    model.add(Dropout(0.1))

    # Block 4 — ReLU
    model.add(Conv1D(filter_num[4], kernel_size[4], strides=conv_stride[4],
                     padding='same', name='block4_conv1'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Conv1D(filter_num[4], kernel_size[4], strides=conv_stride[4],
                     padding='same', name='block4_conv2'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(MaxPooling1D(pool_size=pool_size_list[4], strides=pool_stride[4],
                           padding='same', name='block4_pool'))
    model.add(Dropout(0.1))

    # Fully Connected
    model.add(Flatten())
    model.add(Dense(512, kernel_initializer=GlorotUniform(seed=0), name='fc1'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(0.7))

    model.add(Dense(512, kernel_initializer=GlorotUniform(seed=0), name='fc2'))
    model.add(BatchNormalization())
    model.add(Activation('relu'))
    model.add(Dropout(0.5))

    model.add(Dense(classes, kernel_initializer=GlorotUniform(seed=0), name='fc3'))
    model.add(Activation('softmax'))

    return model

print("DFNet defined.")

In [ ]:
print("=" * 60)
print("CLOSED WORLD — Loading data from:", CLOSE_DIR)
print("=" * 60)

X_cw, y_cw, cw_label_map = load_dataset_from_dir(CLOSE_DIR, length=LENGTH)
NB_CLASSES_CW = len(cw_label_map)

print(f"Total samples : {X_cw.shape[0]}")
print(f"Total classes : {NB_CLASSES_CW}")
print(f"Feature shape : {X_cw.shape[1]}")
print(f"Label map (first 5): {dict(list(cw_label_map.items())[:5])}")

In [ ]:
# Split: 70% train / 10% valid / 20% test (stratified)
X_train_cw, X_temp, y_train_cw, y_temp = train_test_split(
    X_cw, y_cw,
    test_size=(1 - TRAIN_RATIO),
    stratify=y_cw,
    random_state=SEED
)

# valid_ratio relative to X_temp
valid_rel = VALID_RATIO / (VALID_RATIO + TEST_RATIO)
X_valid_cw, X_test_cw, y_valid_cw, y_test_cw = train_test_split(
    X_temp, y_temp,
    test_size=(1 - valid_rel),
    stratify=y_temp,
    random_state=SEED
)

print(f"Train : {X_train_cw.shape[0]} samples")
print(f"Valid : {X_valid_cw.shape[0]} samples")
print(f"Test  : {X_test_cw.shape[0]} samples")

# Reshape cho CNN: (N, LENGTH, 1)
X_train_cw = X_train_cw[:, :, np.newaxis]
X_valid_cw = X_valid_cw[:, :, np.newaxis]
X_test_cw  = X_test_cw[:, :, np.newaxis]

# One-hot encode
y_train_cw_cat = to_categorical(y_train_cw, NB_CLASSES_CW)
y_valid_cw_cat = to_categorical(y_valid_cw, NB_CLASSES_CW)
y_test_cw_cat  = to_categorical(y_test_cw,  NB_CLASSES_CW)

print("Data prepared for Closed World.")

In [ ]:
print("Building DFNet for Closed World...")
INPUT_SHAPE = (LENGTH, 1)

model_cw = build_dfnet(input_shape=INPUT_SHAPE, classes=NB_CLASSES_CW)
model_cw.compile(
    loss='categorical_crossentropy',
    optimizer=Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999, epsilon=1e-8),
    metrics=['accuracy']
)
model_cw.summary()

In [ ]:
history_cw = model_cw.fit(
    X_train_cw, y_train_cw_cat,
    batch_size=BATCH_SIZE,
    epochs=NB_EPOCH,
    verbose=VERBOSE,
    validation_data=(X_valid_cw, y_valid_cw_cat)
)

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history_cw.history['loss'], label='Train Loss')
axes[0].plot(history_cw.history['val_loss'], label='Val Loss')
axes[0].set_title('Closed World — Loss'); axes[0].legend()
axes[1].plot(history_cw.history['accuracy'], label='Train Acc')
axes[1].plot(history_cw.history['val_accuracy'], label='Val Acc')
axes[1].set_title('Closed World — Accuracy'); axes[1].legend()
plt.tight_layout(); plt.show()

In [ ]:
# Test accuracy
score_cw = model_cw.evaluate(X_test_cw, y_test_cw_cat, verbose=0)
print(f"\nClosed World — Test Loss    : {score_cw[0]:.4f}")
print(f"Closed World — Test Accuracy: {score_cw[1]:.4f}")

# Detailed classification report
y_pred_probs_cw = model_cw.predict(X_test_cw, verbose=0)
y_pred_cw = np.argmax(y_pred_probs_cw, axis=1)
print("\n--- Classification Report (Closed World) ---")
print(classification_report(y_test_cw, y_pred_cw, digits=4))

# Save report
report_dict_cw = classification_report(y_test_cw, y_pred_cw, digits=4, output_dict=True)
report_df_cw = pd.DataFrame(report_dict_cw).transpose()
report_df_cw.to_csv('/kaggle/working/DF_ClosedWorld_Results.csv', float_format='%.4f')
print("Report saved to /kaggle/working/DF_ClosedWorld_Results.csv")

In [ ]:
# Save model
model_cw.save('/kaggle/working/DF_ClosedWorld.h5')
print("Model saved to /kaggle/working/DF_ClosedWorld.h5")

In [ ]:

print("=" * 60)
print("OPEN WORLD — Loading unmonitored data from:", OPEN_DIR)
print("=" * 60)

# Load tất cả unmonitored (label tạm = -1, sẽ gán lại sau)
X_unmon_all, _, _ = load_dataset_from_dir(OPEN_DIR, length=LENGTH, fixed_label=0)

print(f"Total unmonitored samples available: {len(X_unmon_all)}")
assert len(X_unmon_all) >= N_UNKNOWN_TRAIN + N_UNKNOWN_TEST, \
    f"Không đủ data! Cần {N_UNKNOWN_TRAIN + N_UNKNOWN_TEST}, có {len(X_unmon_all)}"

# Shuffle và chọn
idx_unmon = np.random.permutation(len(X_unmon_all))
idx_train_unmon = idx_unmon[:N_UNKNOWN_TRAIN]
idx_test_unmon  = idx_unmon[N_UNKNOWN_TRAIN : N_UNKNOWN_TRAIN + N_UNKNOWN_TEST]

X_unmon_train = X_unmon_all[idx_train_unmon]
X_unmon_test  = X_unmon_all[idx_test_unmon]

print(f"Unknown for train : {len(X_unmon_train)}")
print(f"Unknown for test  : {len(X_unmon_test)}")

In [ ]:
UNMON_LABEL = NB_CLASSES_CW
NB_CLASSES_OW = NB_CLASSES_CW + 1   # monitored classes + 1 unmonitored class

print(f"Open World classes: {NB_CLASSES_OW} ({NB_CLASSES_CW} monitored + 1 unmonitored)")
print(f"Unmonitored label index: {UNMON_LABEL}")

# Tạo label cho unmonitored
y_unmon_train = np.full(len(X_unmon_train), UNMON_LABEL, dtype=np.int32)
y_unmon_test  = np.full(len(X_unmon_test),  UNMON_LABEL, dtype=np.int32)

In [ ]:
X_mon_train_ow, X_mon_temp, y_mon_train_ow, y_mon_temp = train_test_split(
    X_cw, y_cw,
    test_size=(1 - TRAIN_RATIO),
    stratify=y_cw,
    random_state=SEED
)
X_mon_valid_ow, X_mon_test_ow, y_mon_valid_ow, y_mon_test_ow = train_test_split(
    X_mon_temp, y_mon_temp,
    test_size=(1 - valid_rel),
    stratify=y_mon_temp,
    random_state=SEED
)

n_valid_unmon = max(1, int(N_UNKNOWN_TRAIN * (VALID_RATIO / (TRAIN_RATIO + VALID_RATIO))))
n_train_unmon = N_UNKNOWN_TRAIN - n_valid_unmon

X_train_ow = np.concatenate([X_mon_train_ow, X_unmon_train[:n_train_unmon]])
y_train_ow = np.concatenate([y_mon_train_ow, y_unmon_train[:n_train_unmon]])

X_valid_ow = np.concatenate([X_mon_valid_ow, X_unmon_train[n_train_unmon:N_UNKNOWN_TRAIN]])
y_valid_ow = np.concatenate([y_mon_valid_ow, y_unmon_train[n_train_unmon:N_UNKNOWN_TRAIN]])

X_test_Mon  = X_mon_test_ow    # monitored test samples
y_test_Mon  = y_mon_test_ow
X_test_Unmon = X_unmon_test    # unmonitored test samples
y_test_Unmon = y_unmon_test

print(f"OW Train  : {len(X_train_ow)} ({len(X_mon_train_ow)} mon + {n_train_unmon} unmon)")
print(f"OW Valid  : {len(X_valid_ow)} ({len(X_mon_valid_ow)} mon + {N_UNKNOWN_TRAIN - n_train_unmon} unmon)")
print(f"OW Test Mon   : {len(X_test_Mon)}")
print(f"OW Test Unmon : {len(X_test_Unmon)}")

# Shuffle train
idx_shuf = np.random.permutation(len(X_train_ow))
X_train_ow = X_train_ow[idx_shuf]
y_train_ow = y_train_ow[idx_shuf]

# Reshape
X_train_ow   = X_train_ow[:, :, np.newaxis]
X_valid_ow   = X_valid_ow[:, :, np.newaxis]
X_test_Mon   = X_test_Mon[:, :, np.newaxis]
X_test_Unmon = X_test_Unmon[:, :, np.newaxis]

# One-hot
y_train_ow_cat = to_categorical(y_train_ow, NB_CLASSES_OW)
y_valid_ow_cat = to_categorical(y_valid_ow, NB_CLASSES_OW)

print("Data prepared for Open World.")

In [ ]:
print("Building DFNet for Open World...")
model_ow = build_dfnet(input_shape=INPUT_SHAPE, classes=NB_CLASSES_OW)
model_ow.compile(
    loss='categorical_crossentropy',
    optimizer=Adamax(learning_rate=0.002, beta_1=0.9, beta_2=0.999, epsilon=1e-8),
    metrics=['accuracy']
)
model_ow.summary()

In [ ]:
history_ow = model_ow.fit(
    X_train_ow, y_train_ow_cat,
    batch_size=BATCH_SIZE,
    epochs=NB_EPOCH,
    verbose=VERBOSE,
    validation_data=(X_valid_ow, y_valid_ow_cat)
)

In [ ]:
model_ow.save('/kaggle/working/DF_OpenWorld.h5')
print("Model saved to /kaggle/working/DF_OpenWorld.h5")

In [ ]:
# Predict softmax vectors
result_Mon   = model_ow.predict(X_test_Mon,   verbose=0)
result_Unmon = model_ow.predict(X_test_Unmon, verbose=0)

monitored_labels   = list(range(NB_CLASSES_CW))   # 0 .. NB_CLASSES_CW-1
unmonitored_labels = [UNMON_LABEL]

def evaluate_ow(threshold_val, result_Mon, result_Unmon):
    """Tính TP/FP/TN/FN theo threshold như paper gốc."""
    TP = FP = TN = FN = 0

    for sm in result_Mon:
        max_prob = np.max(sm[:-1])   # max prob over monitored classes only
        if max_prob >= threshold_val:
            TP += 1   # đúng là monitored, predict monitored
        else:
            FN += 1   # đúng là monitored, predict unknown

    for sm in result_Unmon:
        max_prob = np.max(sm[:-1])   # max prob over monitored classes
        if max_prob >= threshold_val:
            FP += 1   # đúng là unknown, predict monitored
        else:
            TN += 1   # đúng là unknown, predict unknown

    TPR = TP / (TP + FN) if (TP + FN) > 0 else 0
    FPR = FP / (FP + TN) if (FP + TN) > 0 else 0
    Prec = TP / (TP + FP) if (TP + FP) > 0 else 0
    Rec  = TPR
    return TP, FP, TN, FN, TPR, FPR, Prec, Rec

# Sweep thresholds
thresholds = 1.0 - 1.0 / np.logspace(0.05, 2, num=15, endpoint=True)
rows = []
print(f"{'Threshold':>12} {'TP':>6} {'FP':>6} {'TN':>6} {'FN':>6} {'TPR':>8} {'FPR':>8} {'Prec':>8} {'Rec':>8}")
print("-" * 80)
for th in thresholds:
    TP, FP, TN, FN, TPR, FPR, Prec, Rec = evaluate_ow(th, result_Mon, result_Unmon)
    rows.append({'threshold': th, 'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN,
                 'TPR': TPR, 'FPR': FPR, 'Precision': Prec, 'Recall': Rec})
    print(f"{th:12.6f} {TP:6d} {FP:6d} {TN:6d} {FN:6d} {TPR:8.4f} {FPR:8.4f} {Prec:8.4f} {Rec:8.4f}")

ow_df = pd.DataFrame(rows)
ow_df.to_csv('/kaggle/working/OpenWorld_NoDef.csv', float_format='%.6f', index=False)
print("\nThreshold sweep saved to /kaggle/working/OpenWorld_NoDef.csv")

In [ ]:
# Binary ground truth: monitored=1, unmonitored=0
y_true_binary = np.array([1] * len(result_Mon) + [0] * len(result_Unmon))

# Score = max softmax over monitored classes
score_known = np.concatenate([
    np.max(result_Mon[:, :-1],   axis=1),
    np.max(result_Unmon[:, :-1], axis=1)
])

# ROC
fpr_arr, tpr_arr, roc_thresholds = roc_curve(y_true_binary, score_known)
roc_auc = auc(fpr_arr, tpr_arr)

# PR
prec_arr, rec_arr, pr_thresholds = precision_recall_curve(y_true_binary, score_known)
pr_thresholds_padded = np.append(pr_thresholds, np.nan)

os.makedirs('/kaggle/working/curve_data', exist_ok=True)

pd.DataFrame({
    'fpr': fpr_arr,
    'tpr': tpr_arr,
    'threshold': roc_thresholds
}).to_csv('/kaggle/working/curve_data/roc_curve.csv', index=False)

pd.DataFrame({
    'precision': prec_arr,
    'recall': rec_arr,
    'threshold': pr_thresholds_padded
}).to_csv('/kaggle/working/curve_data/precision_recall_curve.csv', index=False)

# Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(fpr_arr, tpr_arr, lw=2, label=f'ROC (AUC = {roc_auc:.4f})')
axes[0].plot([0,1],[0,1],'k--')
axes[0].set_xlabel('FPR'); axes[0].set_ylabel('TPR')
axes[0].set_title('Open World — ROC Curve'); axes[0].legend()

axes[1].plot(rec_arr, prec_arr, lw=2, color='darkorange')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision')
axes[1].set_title('Open World — Precision-Recall Curve')

plt.tight_layout(); plt.show()
print(f"ROC AUC: {roc_auc:.4f}")
print(f"ROC curve     : {len(fpr_arr)} points → /kaggle/working/curve_data/roc_curve.csv")
print(f"PR curve      : {len(prec_arr)} points → /kaggle/working/curve_data/precision_recall_curve.csv")

In [ ]:
TARGET_THRESHOLD = 0.5

y_true_ow = np.concatenate([y_test_Mon, y_test_Unmon])
y_pred_ow = []

for sm in np.concatenate([result_Mon, result_Unmon]):
    max_prob = np.max(sm[:-1])   # max prob over monitored classes
    if max_prob >= TARGET_THRESHOLD:
        y_pred_ow.append(np.argmax(sm[:-1]))   # predicted monitored class
    else:
        y_pred_ow.append(UNMON_LABEL)           # predicted unknown

y_pred_ow = np.array(y_pred_ow)

print(f"--- Classification Report (Open World, threshold={TARGET_THRESHOLD}) ---")
print(classification_report(y_true_ow, y_pred_ow, digits=4))

report_ow = classification_report(y_true_ow, y_pred_ow, digits=4, output_dict=True)
pd.DataFrame(report_ow).transpose().to_csv(
    f'/kaggle/working/OW_ClassReport_th{TARGET_THRESHOLD:.2f}.csv', float_format='%.4f')
print(f"Report saved.")